# Mini ARC-AGI 3: Reasoning Benchmark for DeepSeek-R1-Distill-Qwen-1.5B

## Overview

This notebook evaluates **DeepSeek-R1-Distill-Qwen-1.5B** on a custom benchmark inspired by **ARC-AGI 3**, where frontier models score under 2%.

### Design Principles

1. **No instructions** — The model receives only example input/output grid pairs and must infer the transformation rule entirely from examples.
2. **Forced reasoning** — Every puzzle requires multi-step abstract reasoning (spatial, logical, compositional). Simple pattern matching is insufficient.
3. **Procedural generation** — Puzzles are generated programmatically, ensuring diversity and consistent difficulty.
4. **Verifiable answers** — Each puzzle has a deterministic correct answer computed by the generator.
5. **Chain-of-thought activation** — DeepSeek-R1 uses `<think>...</think>` reasoning blocks. Puzzles are designed so that the model *must* reason step-by-step to succeed.

### Puzzle Categories (12 types × 3 instances = 36 puzzles)

| # | Puzzle | Reasoning Required |
|---|--------|-------------------|
| 1 | Gravity + Column Sort | Physics simulation + sorting |
| 2 | Maze Path (BFS) | Pathfinding + spatial reasoning |
| 3 | Symmetry Completion | Pattern recognition + spatial completion |
| 4 | Region Flood Coloring | Flood fill + area ranking |
| 5 | Largest Object Replication | Connected components + counting + replication |
| 6 | Tile Pattern Extrapolation | Pattern recognition + 2D tiling |
| 7 | Color Chain Transform | Mapping inference + application |
| 8 | Block Expansion | Spatial expansion + arithmetic |
| 9 | Conditional Transform | Counting + conditional logic + spatial transform |
| 10 | Diagonal Fill | Diagonal reasoning + spatial |
| 11 | Row Extremes Marking | Row scanning + comparison logic |
| 12 | Shape Sort & Arrange | Shape identification + sorting + arrangement |

Each puzzle provides **3 example pairs** and **1 test input**. The model must output the correct transformed grid.

> **Note**: Enable GPU accelerator (T4 x2 or P100) and Internet access in Kaggle settings.

In [ ]:
!pip install -q transformers torch accelerate
!pip install -q matplotlib seaborn

In [ ]:
import torch
import random
import json
import re
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from collections import deque, Counter
from copy import deepcopy
from transformers import AutoModelForCausalLM, AutoTokenizer

# ── Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Configuration ──
MODEL_NAME           = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
MAX_NEW_TOKENS       = 8192      # generous budget for reasoning
NUM_EXAMPLES         = 3         # example pairs per puzzle
NUM_PUZZLES_PER_TYPE = 3         # puzzles per type
# Greedy decoding (do_sample=False) for deterministic, reproducible outputs.
# We do NOT pass temperature/top_p since they are ignored (and cause warnings)
# when do_sample=False.

print("Configuration loaded.")
print(f"Model: {MODEL_NAME}")
print(f"Max new tokens: {MAX_NEW_TOKENS}")
print(f"Puzzles per type: {NUM_PUZZLES_PER_TYPE}")
print(f"Total puzzle types: 12 → {12 * NUM_PUZZLES_PER_TYPE} puzzles total")
print(f"Decoding: greedy (do_sample=False)")

## 1. Load DeepSeek-R1-Distill-Qwen-1.5B

DeepSeek-R1-Distill-Qwen-1.5B is a 1.5B parameter reasoning model distilled from DeepSeek-R1.
It uses `<think>...</think>` tags for chain-of-thought reasoning before producing final answers.

In [ ]:
print(f"Loading {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"✓ Model loaded on {model.device}")
print(f"  Parameters: {n_params / 1e9:.2f}B")
print(f"  Dtype: {next(model.parameters()).dtype}")

## 2. Grid Utilities & Helpers

ARC-AGI style grids are 2D integer arrays (values 0–9). These helpers support generation, visualization, and evaluation.

In [ ]:
# ============================================================
# Grid Utilities
# ============================================================

def empty_grid(h, w, val=0):
    """Create an h×w grid filled with *val*."""
    return [[val] * w for _ in range(h)]

def grid_copy(g):
    return [row[:] for row in g]

def grid_dims(g):
    return len(g), len(g[0])

def grid_to_str(g):
    """Pretty-print grid as 2D array string for prompts."""
    rows = ['[' + ','.join(str(c) for c in row) + ']' for row in g]
    return '[' + ',\n '.join(rows) + ']'

def count_nonzero(g):
    return sum(1 for row in g for c in row if c != 0)

def distinct_colors(g):
    return set(c for row in g for c in row if c != 0)

def connected_components(g, connectivity=4):
    """Find connected components of same-color non-zero cells."""
    h, w = grid_dims(g)
    visited = empty_grid(h, w, False)
    comps = []
    nbrs = [(-1,0),(1,0),(0,-1),(0,1)] if connectivity==4 else [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]
    for r in range(h):
        for c in range(w):
            if g[r][c] != 0 and not visited[r][c]:
                color = g[r][c]
                comp = []
                q = deque([(r, c)])
                visited[r][c] = True
                while q:
                    cr, cc = q.popleft()
                    comp.append((cr, cc))
                    for dr, dc in nbrs:
                        nr, nc = cr+dr, cc+dc
                        if 0<=nr<h and 0<=nc<w and not visited[nr][nc] and g[nr][nc]==color:
                            visited[nr][nc] = True
                            q.append((nr, nc))
                comps.append({'color': color, 'cells': comp, 'size': len(comp)})
    return comps

def bounding_box(cells):
    rs = [r for r,c in cells]
    cs = [c for r,c in cells]
    return min(rs), min(cs), max(rs), max(cs)

# ============================================================
# Visualization
# ============================================================

_CMAP = mcolors.ListedColormap([
    '#1a1a2e', '#e63946', '#06d6a0', '#118ab2', '#ffd166',
    '#ef476f', '#7209b7', '#f77f00', '#888888', '#f0f0f0'
])

def visualize_grid(g, ax=None, title=""):
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(3, 3))
    h, w = grid_dims(g)
    ax.imshow(np.array(g), cmap=_CMAP, vmin=0, vmax=9)
    ax.set_xticks(range(w))
    ax.set_yticks(range(h))
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.set_title(title, fontsize=9)
    return ax

def visualize_puzzle(examples, test_input, test_output=None, model_output=None):
    n_ex = len(examples)
    n_cols = 2 + n_ex * 2 + (2 if model_output else 0)
    fig, axes = plt.subplots(1, n_cols, figsize=(n_cols * 2.2, 2.5))
    if n_cols == 1:
        axes = [axes]
    idx = 0
    for i, (inp, out) in enumerate(examples):
        visualize_grid(inp, axes[idx], f"Ex{i+1} In")
        idx += 1
        visualize_grid(out, axes[idx], f"Ex{i+1} Out")
        idx += 1
    visualize_grid(test_input, axes[idx], "Test In")
    idx += 1
    if test_output is not None:
        visualize_grid(test_output, axes[idx], "Expected")
        idx += 1
    if model_output is not None:
        visualize_grid(model_output, axes[idx], "Model")
    plt.tight_layout()
    return fig

print("Grid utilities loaded ✓")

## 3. Puzzle Generators

Each puzzle type has:
- `gen_params(rng)` → rule parameters (shared across all examples in a puzzle)
- `gen_instance(params, rng)` → a single (input, output) grid pair

A **puzzle** = 3 example pairs (same rule, different grids) + 1 test pair.

This ensures the model must **infer the rule** from examples, not memorize a fixed transformation.

In [ ]:
# ============================================================
# PUZZLE TYPE 1: Gravity + Column Sort
# Rule: Non-zero cells fall to bottom of each column (gravity).
#        Then columns are sorted left→right by their bottom-most value.
# Reasoning: Two-step transformation (physics + sorting).
# ============================================================

def gravity_sort_params(rng):
    return {}

def gravity_sort_instance(params, rng):
    h, w = rng.randint(5, 8), rng.randint(5, 8)
    grid = empty_grid(h, w)
    for _ in range(rng.randint(5, min(h * w // 3, 14))):
        r, c = rng.randint(0, h-1), rng.randint(0, w-1)
        grid[r][c] = rng.randint(1, 4)
    # Step 1: gravity
    fallen = empty_grid(h, w)
    for c in range(w):
        col = [grid[r][c] for r in range(h) if grid[r][c] != 0]
        for i, v in enumerate(col):
            fallen[h - len(col) + i][c] = v
    # Step 2: sort columns by bottom value
    cols = []
    for c in range(w):
        col = [fallen[r][c] for r in range(h)]
        bottom = next((v for v in reversed(col) if v != 0), 0)
        cols.append((bottom, c, col))
    cols.sort(key=lambda x: (x[0], x[1]))
    output = empty_grid(h, w)
    for new_c, (_, _, col) in enumerate(cols):
        for r in range(h):
            output[r][new_c] = col[r]
    return grid, output


# ============================================================
# PUZZLE TYPE 2: Maze Path (BFS Shortest Path)
# Rule: 1=wall, 2=start, 3=end. Mark shortest path with 4.
# Reasoning: Pathfinding, spatial navigation, BFS.
# ============================================================

def maze_path_params(rng):
    return {}

def maze_path_instance(params, rng):
    for _ in range(50):
        h, w = rng.randint(6, 9), rng.randint(6, 9)
        grid = empty_grid(h, w, 0)
        for _ in range(rng.randint(h*w//4, h*w//3)):
            grid[rng.randint(0, h-1)][rng.randint(0, w-1)] = 1
        sr, sc, er, ec = 0, 0, h-1, w-1
        grid[sr][sc] = 2
        grid[er][ec] = 3
        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            for r, c in [(sr+dr, sc+dc), (er+dr, ec+dc)]:
                if 0<=r<h and 0<=c<w and grid[r][c] == 1:
                    grid[r][c] = 0
        # BFS
        q = deque([(sr, sc)])
        visited = {(sr, sc)}
        parent = {(sr, sc): None}
        found = False
        while q:
            r, c = q.popleft()
            if (r, c) == (er, ec):
                found = True
                break
            for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                nr, nc = r+dr, c+dc
                if 0<=nr<h and 0<=nc<w and (nr,nc) not in visited and grid[nr][nc] != 1:
                    visited.add((nr, nc))
                    parent[(nr, nc)] = (r, c)
                    q.append((nr, nc))
        if not found:
            continue
        # Reconstruct path
        path = []
        cur = (er, ec)
        while cur is not None:
            path.append(cur)
            cur = parent[cur]
        path.reverse()
        output = grid_copy(grid)
        for r, c in path[1:-1]:
            output[r][c] = 4
        return grid, output
    # Fallback: simple open grid
    h, w = 6, 6
    grid = empty_grid(h, w, 0)
    grid[0][0] = 2
    grid[5][5] = 3
    output = grid_copy(grid)
    for i in range(1, 5):
        output[0][i] = 4
    for i in range(1, 5):
        output[i][5] = 4
    return grid, output


# ============================================================
# PUZZLE TYPE 3: Symmetry Completion
# Rule: Complete the grid to be symmetric along an axis.
#        Axis (h/v/d1/d2) is constant across examples — must be inferred.
# Reasoning: Recognize symmetry type, complete missing cells.
# ============================================================

def symmetry_params(rng):
    return {'axis': rng.choice(['h', 'v', 'd1', 'd2'])}

def symmetry_instance(params, rng):
    axis = params['axis']
    if axis in ('d1', 'd2'):
        s = rng.randint(4, 6)
        h, w = s, s
    else:
        h, w = rng.randint(4, 7), rng.randint(4, 7)
    full = empty_grid(h, w)
    colors = rng.sample(range(1, 6), rng.randint(2, 4))
    for _ in range(rng.randint(4, 9)):
        r, c = rng.randint(0, h-1), rng.randint(0, w-1)
        v = rng.choice(colors)
        full[r][c] = v
        if axis == 'h':
            full[h-1-r][c] = v
        elif axis == 'v':
            full[r][w-1-c] = v
        elif axis == 'd1':
            full[c][r] = v
        elif axis == 'd2':
            full[h-1-c][w-1-r] = v
    # Remove cells to create input
    inp = grid_copy(full)
    nz = [(r, c) for r in range(h) for c in range(w) if full[r][c] != 0]
    rng.shuffle(nz)
    n_remove = min(len(nz) // 2, rng.randint(2, 5))
    for r, c in nz[:n_remove]:
        inp[r][c] = 0
    return inp, full


# ============================================================
# PUZZLE TYPE 4: Region Flood Coloring
# Rule: Grid has boundary walls (color 1). Enclosed regions are colored
#        by area rank: smallest→2, next→3, etc.
# Reasoning: Flood fill, area computation, ranking.
# ============================================================

def region_color_params(rng):
    return {}

def region_color_instance(params, rng):
    h, w = rng.randint(6, 9), rng.randint(6, 9)
    grid = empty_grid(h, w, 0)
    # Draw random boundary lines
    n_lines = rng.randint(2, 4)
    for _ in range(n_lines):
        if rng.random() < 0.5:
            r = rng.randint(1, h-2)
            c1, c2 = sorted(rng.sample(range(w), 2))
            for c in range(c1, c2+1):
                grid[r][c] = 1
        else:
            c = rng.randint(1, w-2)
            r1, r2 = sorted(rng.sample(range(h), 2))
            for r in range(r1, r2+1):
                grid[r][c] = 1
    # Flood fill non-wall regions
    visited = empty_grid(h, w, False)
    regions = []
    for r in range(h):
        for c in range(w):
            if grid[r][c] == 0 and not visited[r][c]:
                cells = []
                q = deque([(r, c)])
                visited[r][c] = True
                while q:
                    cr, cc = q.popleft()
                    cells.append((cr, cc))
                    for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                        nr, nc = cr+dr, cc+dc
                        if 0<=nr<h and 0<=nc<w and not visited[nr][nc] and grid[nr][nc]==0:
                            visited[nr][nc] = True
                            q.append((nr, nc))
                regions.append(cells)
    # Sort by area, assign colors
    regions.sort(key=len)
    output = grid_copy(grid)
    for i, cells in enumerate(regions):
        color = min(i + 2, 9)
        for r, c in cells:
            output[r][c] = color
    return grid, output


# ============================================================
# PUZZLE TYPE 5: Largest Object Replication
# Rule: Find largest connected component. Count distinct colors in grid.
#        Replicate largest object horizontally N times (N = distinct color count).
# Reasoning: Connected components, counting, replication.
# ============================================================

def largest_obj_params(rng):
    return {}

def largest_obj_instance(params, rng):
    h, w = rng.randint(4, 6), rng.randint(4, 6)
    grid = empty_grid(h, w)
    # Place a few objects
    n_objects = rng.randint(2, 4)
    for _ in range(n_objects):
        color = rng.randint(1, 5)
        obj_h, obj_w = rng.randint(1, 3), rng.randint(1, 3)
        r0, c0 = rng.randint(0, h-obj_h), rng.randint(0, w-obj_w)
        for dr in range(obj_h):
            for dc in range(obj_w):
                if rng.random() < 0.7:
                    grid[r0+dr][c0+dc] = color
    comps = connected_components(grid)
    if not comps:
        grid[rng.randint(0,h-1)][rng.randint(0,w-1)] = 2
        comps = connected_components(grid)
    largest = max(comps, key=lambda x: x['size'])
    # Bounding box of largest
    rmin, cmin, rmax, cmax = bounding_box(largest['cells'])
    obj_h = rmax - rmin + 1
    obj_w = cmax - cmin + 1
    obj_color = largest['color']
    # Extract object pattern
    pattern = empty_grid(obj_h, obj_w, 0)
    for r, c in largest['cells']:
        pattern[r-rmin][c-cmin] = obj_color
    n_colors = len(distinct_colors(grid))
    n_rep = max(n_colors, 1)
    out_w = obj_w * n_rep + (n_rep - 1)
    output = empty_grid(obj_h, out_w, 0)
    for rep in range(n_rep):
        offset = rep * (obj_w + 1)
        for r in range(obj_h):
            for c in range(obj_w):
                output[r][offset + c] = pattern[r][c]
    return grid, output


# ============================================================
# PUZZLE TYPE 6: Tile Pattern Extrapolation
# Rule: A small tile is repeated to fill the grid. Given a partial
#        grid (one tile + partial), complete the full grid.
#        Tile size is constant across examples — must be inferred.
# Reasoning: Pattern recognition, 2D tiling, modular arithmetic.
# ============================================================

def tile_pattern_params(rng):
    th, tw = rng.randint(2, 3), rng.randint(2, 3)
    return {'tile_h': th, 'tile_w': tw}

def tile_pattern_instance(params, rng):
    th, tw = params['tile_h'], params['tile_w']
    reps_h, reps_w = rng.randint(2, 3), rng.randint(2, 3)
    h, w = th * reps_h, tw * reps_w
    tile = empty_grid(th, tw, 0)
    for r in range(th):
        for c in range(tw):
            tile[r][c] = rng.choice([0, 0, rng.randint(1, 5)])
    # Full grid = tile repeated
    full = empty_grid(h, w, 0)
    for r in range(h):
        for c in range(w):
            full[r][c] = tile[r % th][c % tw]
    # Input: show only first tile + partial second row/column
    inp = empty_grid(h, w, 0)
    # Show first tile completely
    for r in range(th):
        for c in range(tw):
            inp[r][c] = full[r][c]
    # Show partial: first row of second tile repetition
    for c in range(min(tw, w)):
        if th < h:
            inp[th][c] = full[th][c]
    # Show first column of second horizontal repetition
    for r in range(min(th, h)):
        if tw < w:
            inp[r][tw] = full[r][tw]
    return inp, full


print("Puzzle generators 1–6 loaded ✓")

In [ ]:
# ============================================================
# PUZZLE TYPE 7: Color Chain Transform
# Rule: Each color maps to the next in a circular chain.
#        e.g. 1→3→2→4→1. Chain must be inferred from examples.
# Reasoning: Mapping inference, chain reasoning, application.
# ============================================================

def color_chain_params(rng):
    colors = rng.sample(range(1, 6), rng.randint(3, 4))
    chain = colors[:] + [colors[0]]  # circular
    mapping = {}
    for i in range(len(colors)):
        mapping[colors[i]] = chain[i+1]
    return {'mapping': mapping}

def color_chain_instance(params, rng):
    mapping = params['mapping']
    h, w = rng.randint(4, 7), rng.randint(4, 7)
    grid = empty_grid(h, w)
    for _ in range(rng.randint(5, h*w//2)):
        r, c = rng.randint(0, h-1), rng.randint(0, w-1)
        grid[r][c] = rng.choice(list(mapping.keys()))
    output = empty_grid(h, w)
    for r in range(h):
        for c in range(w):
            output[r][c] = mapping.get(grid[r][c], 0)
    return grid, output


# ============================================================
# PUZZLE TYPE 8: Block Expansion
# Rule: Each non-zero cell expands into a square block of that color.
#        Block size = cell value. Grid scales accordingly.
#        Expansion factor is constant across examples.
# Reasoning: Spatial expansion, arithmetic, coordinate mapping.
# ============================================================

def block_expansion_params(rng):
    return {}

def block_expansion_instance(params, rng):
    h, w = rng.randint(3, 5), rng.randint(3, 5)
    grid = empty_grid(h, w)
    for _ in range(rng.randint(2, 5)):
        r, c = rng.randint(0, h-1), rng.randint(0, w-1)
        grid[r][c] = rng.randint(2, 3)  # block size 2 or 3
    scale = max(grid[r][c] for r in range(h) for c in range(w) if grid[r][c] != 0)
    out_h, out_w = h * scale, w * scale
    output = empty_grid(out_h, out_w, 0)
    for r in range(h):
        for c in range(w):
            v = grid[r][c]
            if v != 0:
                for dr in range(v):
                    for dc in range(v):
                        output[r*scale+dr][c*scale+dc] = v
    return grid, output


# ============================================================
# PUZZLE TYPE 9: Conditional Transform
# Rule: Count connected components.
#        If odd → reflect horizontally (left-right mirror).
#        If even → reflect vertically (top-bottom mirror).
#        If zero → rotate 180°.
# Reasoning: Counting, conditional logic, spatial transformation.
# ============================================================

def conditional_transform_params(rng):
    return {}

def conditional_transform_instance(params, rng):
    h, w = rng.randint(4, 7), rng.randint(4, 7)
    grid = empty_grid(h, w)
    # Place objects
    n_obj_target = rng.randint(1, 5)
    for _ in range(n_obj_target):
        color = rng.randint(1, 5)
        r, c = rng.randint(0, h-1), rng.randint(0, w-1)
        grid[r][c] = color
        # Maybe make it 2-cell
        if rng.random() < 0.4:
            dr, dc = rng.choice([(-1,0),(1,0),(0,-1),(0,1)])
            nr, nc = r+dr, c+dc
            if 0<=nr<h and 0<=nc<w:
                grid[nr][nc] = color
    comps = connected_components(grid)
    n = len(comps)
    if n == 0:
        output = [row[::-1] for row in grid[::-1]]  # rotate 180
    elif n % 2 == 1:
        output = [row[::-1] for row in grid]  # horizontal mirror
    else:
        output = grid[::-1]  # vertical mirror
    return grid, output


# ============================================================
# PUZZLE TYPE 10: Diagonal Fill
# Rule: For each non-zero cell (in order of appearance, row-major),
#        fill both diagonals passing through it with that cell's color.
#        Later cells overwrite earlier ones. Original cells preserved.
# Reasoning: Diagonal reasoning, spatial, ordering.
# ============================================================

def diagonal_fill_params(rng):
    return {}

def diagonal_fill_instance(params, rng):
    h, w = rng.randint(5, 8), rng.randint(5, 8)
    grid = empty_grid(h, w)
    n_cells = rng.randint(2, 4)
    for _ in range(n_cells):
        r, c = rng.randint(0, h-1), rng.randint(0, w-1)
        grid[r][c] = rng.randint(1, 5)
    output = grid_copy(grid)
    # Process in row-major order
    for r in range(h):
        for c in range(w):
            v = grid[r][c]
            if v != 0:
                # Main diagonal
                d = r - c
                for i in range(max(0, d), min(h, w + d)):
                    rr, cc = i, i - d
                    if 0 <= cc < w and (rr, cc) != (r, c):
                        output[rr][cc] = v
                # Anti-diagonal
                s = r + c
                for i in range(max(0, s - w + 1), min(h, s + 1)):
                    rr, cc = i, s - i
                    if 0 <= cc < w and (rr, cc) != (r, c):
                        output[rr][cc] = v
    return grid, output


# ============================================================
# PUZZLE TYPE 11: Row Extremes Marking
# Rule: For each row, find leftmost and rightmost non-zero cell.
#        Mark leftmost with 8, rightmost with 9. If only one non-zero,
#        mark it with 8. Everything else becomes 0.
# Reasoning: Row scanning, comparison, conditional logic.
# ============================================================

def row_extremes_params(rng):
    return {}

def row_extremes_instance(params, rng):
    h, w = rng.randint(4, 7), rng.randint(5, 8)
    grid = empty_grid(h, w)
    for r in range(h):
        if rng.random() < 0.7:
            n_in_row = rng.randint(1, 4)
            cols = rng.sample(range(w), min(n_in_row, w))
            for c in cols:
                grid[r][c] = rng.randint(1, 5)
    output = empty_grid(h, w, 0)
    for r in range(h):
        nz = [c for c in range(w) if grid[r][c] != 0]
        if nz:
            output[r][nz[0]] = 8
            if len(nz) > 1:
                output[r][nz[-1]] = 9
    return grid, output


# ============================================================
# PUZZLE TYPE 12: Shape Sort & Arrange
# Rule: Identify all objects (connected components). Sort by area
#        (smallest first). Place them in a single row, separated by
#        one empty column. Objects are extracted by bounding box.
# Reasoning: Shape identification, sorting, spatial arrangement.
# ============================================================

def shape_sort_params(rng):
    return {}

def shape_sort_instance(params, rng):
    h, w = rng.randint(5, 7), rng.randint(5, 7)
    grid = empty_grid(h, w)
    # Place 2-4 distinct objects with gaps
    n_obj = rng.randint(2, 4)
    for _ in range(30):
        temp = grid_copy(grid)
        color = rng.randint(1, 5)
        oh, ow = rng.randint(1, 3), rng.randint(1, 3)
        r0, c0 = rng.randint(0, h-oh), rng.randint(0, w-ow)
        ok = True
        cells = []
        for dr in range(oh):
            for dc in range(ow):
                if rng.random() < 0.8:
                    if temp[r0+dr][c0+dc] != 0:
                        ok = False
                    cells.append((r0+dr, c0+dc))
        if ok and cells:
            for r, c in cells:
                temp[r][c] = color
            grid = temp
            n_obj -= 1
            if n_obj <= 0:
                break
    comps = connected_components(grid)
    if len(comps) < 2:
        # Ensure at least 2 objects
        for _ in range(10):
            r, c = rng.randint(0, h-1), rng.randint(0, w-1)
            if grid[r][c] == 0:
                grid[r][c] = rng.randint(1, 5)
                break
        comps = connected_components(grid)
    comps.sort(key=lambda x: x['size'])
    # Build output row
    parts = []
    for comp in comps:
        rmin, cmin, rmax, cmax = bounding_box(comp['cells'])
        oh, ow = rmax-rmin+1, cmax-cmin+1
        obj = empty_grid(oh, ow, 0)
        for r, c in comp['cells']:
            obj[r-rmin][c-cmin] = comp['color']
        parts.append(obj)
    max_h = max(grid_dims(p)[0] for p in parts)
    total_w = sum(grid_dims(p)[1] for p in parts) + len(parts) - 1
    output = empty_grid(max_h, total_w, 0)
    offset = 0
    for p in parts:
        ph, pw = grid_dims(p)
        for r in range(ph):
            for c in range(pw):
                output[r][offset+c] = p[r][c]
        offset += pw + 1
    return grid, output


print("Puzzle generators 7–12 loaded ✓")

## 4. Puzzle Registry & Dataset Assembly

Register all 12 puzzle types and generate the full benchmark dataset.

In [ ]:
# ============================================================
# Puzzle Registry
# ============================================================

PUZZLE_TYPES = [
    {'name': 'gravity_sort',         'desc': 'Gravity + Column Sort',          'gen_params': gravity_sort_params,         'gen_instance': gravity_sort_instance},
    {'name': 'maze_path',            'desc': 'Maze Path (BFS)',                'gen_params': maze_path_params,            'gen_instance': maze_path_instance},
    {'name': 'symmetry_completion',  'desc': 'Symmetry Completion',            'gen_params': symmetry_params,             'gen_instance': symmetry_instance},
    {'name': 'region_coloring',      'desc': 'Region Flood Coloring',          'gen_params': region_color_params,         'gen_instance': region_color_instance},
    {'name': 'largest_replication',  'desc': 'Largest Object Replication',     'gen_params': largest_obj_params,          'gen_instance': largest_obj_instance},
    {'name': 'tile_extrapolation',   'desc': 'Tile Pattern Extrapolation',     'gen_params': tile_pattern_params,         'gen_instance': tile_pattern_instance},
    {'name': 'color_chain',          'desc': 'Color Chain Transform',          'gen_params': color_chain_params,          'gen_instance': color_chain_instance},
    {'name': 'block_expansion',      'desc': 'Block Expansion',                'gen_params': block_expansion_params,      'gen_instance': block_expansion_instance},
    {'name': 'conditional_transform','desc': 'Conditional Transform',          'gen_params': conditional_transform_params,'gen_instance': conditional_transform_instance},
    {'name': 'diagonal_fill',        'desc': 'Diagonal Fill',                  'gen_params': diagonal_fill_params,        'gen_instance': diagonal_fill_instance},
    {'name': 'row_extremes',         'desc': 'Row Extremes Marking',           'gen_params': row_extremes_params,         'gen_instance': row_extremes_instance},
    {'name': 'shape_sort',           'desc': 'Shape Sort & Arrange',           'gen_params': shape_sort_params,           'gen_instance': shape_sort_instance},
]

def generate_puzzle(ptype, rng, n_examples=NUM_EXAMPLES):
    """Generate one complete puzzle with examples and test."""
    params = ptype['gen_params'](rng)
    examples = []
    for _ in range(n_examples):
        inp, out = ptype['gen_instance'](params, rng)
        examples.append({'input': inp, 'output': out})
    test_inp, test_out = ptype['gen_instance'](params, rng)
    return {
        'type': ptype['name'],
        'description': ptype['desc'],
        'params': str(params),
        'examples': examples,
        'test_input': test_inp,
        'test_output': test_out,
    }

def generate_dataset(n_per_type=NUM_PUZZLES_PER_TYPE, seed=SEED):
    """Generate the full benchmark dataset."""
    rng = random.Random(seed)
    dataset = []
    for ptype in PUZZLE_TYPES:
        for i in range(n_per_type):
            puzzle = generate_puzzle(ptype, rng)
            puzzle['id'] = f"{ptype['name']}_{i+1}"
            dataset.append(puzzle)
    return dataset

# Generate dataset
dataset = generate_dataset()
print(f"Dataset generated: {len(dataset)} puzzles")
print(f"Puzzle types: {len(PUZZLE_TYPES)}")
for p in dataset[:3]:
    h, w = grid_dims(p['test_input'])
    oh, ow = grid_dims(p['test_output'])
    print(f"  {p['id']}: {p['description']} | input {h}×{w} → output {oh}×{ow}")

## 5. Prompt Construction

The prompt gives the model **no instructions** about the transformation rule.
It only shows example input→output pairs and asks for the test output.

This forces the model to:
1. Infer the rule from examples
2. Reason through the rule step-by-step (using `<think>` blocks — DeepSeek-R1's native reasoning format)
3. Apply the rule to the test input

**Important**: We use the tokenizer's `apply_chat_template` to format the prompt correctly for DeepSeek-R1-Distill-Qwen. This ensures the model generates its native `<think>reasoning</think>answer` structure, activating its full reasoning capabilities. We do **not** pre-fill the `<think>` tag — the model generates it naturally.

In [ ]:
def build_prompt(puzzle):
    """Build a zero-instruction prompt from puzzle examples.

    The prompt gives NO description of the transformation rule.
    The model must infer the rule purely from example input→output pairs.
    """
    lines = []
    lines.append("You are an abstract reasoning system. You will be given example input-output grid pairs that demonstrate a transformation rule. You must infer the rule from the examples and apply it to the test input.")
    lines.append("")
    lines.append("Grids are 2D arrays of integers 0-9. 0 represents empty/background.")
    lines.append("")
    lines.append("=== EXAMPLES ===")
    lines.append("")
    for i, ex in enumerate(puzzle['examples']):
        lines.append(f"--- Example {i+1} ---")
        lines.append("Input:")
        lines.append(grid_to_str(ex['input']))
        lines.append("Output:")
        lines.append(grid_to_str(ex['output']))
        lines.append("")
    lines.append("=== TEST ===")
    lines.append("")
    lines.append("Input:")
    lines.append(grid_to_str(puzzle['test_input']))
    lines.append("")
    lines.append("Apply the transformation rule and output ONLY the resulting grid as a 2D array.")
    lines.append("Think step by step about what rule transforms the inputs into the outputs, then apply it to the test input.")
    return '\n'.join(lines)

# Show an example prompt
sample_prompt = build_prompt(dataset[0])
print(f"Prompt length: {len(sample_prompt)} chars")
print("=" * 60)
print(sample_prompt[:1500])
print("...")

## 6. Model Inference & Grid Parsing

Run DeepSeek-R1 on each puzzle. The model generates reasoning in `midt...<` blocks, then produces a final answer.

**Key parsing details**:
- We decode with `skip_special_tokens=False` so the `midt`/`<` reasoning tags are preserved (they are special tokens for DeepSeek-R1-Distill and would otherwise be stripped).
- `extract_thinking` pulls the reasoning out of the `midt...<` block.
- `parse_grid` is applied **only to the answer text** (after `<`), not the full output. This prevents accidentally extracting example grids the model references while reasoning.

In [ ]:
def parse_grid(text):
    """Extract a 2D integer grid from model output text.

    Uses bracket-depth matching to find the last complete 2D array,
    then falls back to individual row extraction.
    """
    # Strategy 1: bracket-depth scan for 2D arrays ([[...]])
    last_2d = None
    i = 0
    while i < len(text):
        if text[i] == '[':
            depth = 0
            for j in range(i, len(text)):
                if text[j] == '[':
                    depth += 1
                elif text[j] == ']':
                    depth -= 1
                    if depth == 0:
                        candidate = text[i:j+1]
                        # Must be 2D: contains at least one nested [ ]
                        if candidate.count('[') >= 3:
                            last_2d = candidate
                        i = j
                        break
            else:
                break
        i += 1

    if last_2d:
        rows = re.findall(r'\[\s*([\d\s,]+)\]', last_2d)
        grid = []
        for row_str in rows:
            nums = re.findall(r'\d+', row_str)
            if nums:
                grid.append([int(x) for x in nums])
        if grid:
            return grid

    # Strategy 2: collect individual row arrays [d,d,...]
    row_matches = re.findall(r'\[\s*\d+[\s,\d]*\]', text)
    if row_matches:
        grid = []
        for row_str in row_matches:
            nums = re.findall(r'\d+', row_str)
            if nums:
                grid.append([int(x) for x in nums])
        if grid:
            return grid

    return None


def extract_thinking(text):
    """Extract the reasoning from <think>...</think> tags.

    DeepSeek-R1-Distill generates its own <think> tag at the start of
    the response, followed by reasoning, then </think>, then the answer.
    """
    # Standard case: model generated <think>...reasoning...</think>
    match = re.search(r'<think>(.*?)</think>', text, re.DOTALL)
    if match:
        return match.group(1).strip()
    # If there's a <think> but no closing tag, take everything after <think>
    # up to the first grid-looking bracket
    match2 = re.search(r'<think>(.*?)(?:\[|$)', text, re.DOTALL)
    if match2:
        return match2.group(1).strip()
    # If no <think> tag at all, there's no reasoning
    return ""


def run_inference(puzzle, max_new_tokens=MAX_NEW_TOKENS):
    """Run model inference on a single puzzle.

    Uses the tokenizer's chat template so the model generates proper
    <think>...</think> reasoning blocks before its final answer.
    """
    user_prompt = build_prompt(puzzle)

    # Use the chat template — this is the correct way to prompt
    # DeepSeek-R1-Distill-Qwen models. The template adds the proper
    # special tokens and generation prompt, causing the model to
    # naturally produce <think>reasoning</think>answer.
    messages = [{"role": "user", "content": user_prompt}]
    chat_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # greedy decoding for determinism
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens (exclude the prompt).
    # IMPORTANT: skip_special_tokens=False so that <think>/</think> tags
    # are PRESERVED in the decoded text. These are special tokens for
    # DeepSeek-R1-Distill — with skip_special_tokens=True they would be
    # stripped, making it impossible to separate reasoning from the answer.
    input_len = inputs['input_ids'].shape[1]
    full_text = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=False)

    # Remove trailing EOS token (e.g. <｜end▁of▁sentence｜>) if present
    eos_token = tokenizer.eos_token
    if eos_token and eos_token in full_text:
        full_text = full_text.replace(eos_token, '')
    full_text = full_text.strip()

    # Extract the reasoning from the <think>...</think> block
    thinking = extract_thinking(full_text)

    # Parse the grid ONLY from the answer (after </think>), NOT from the
    # full text. The model frequently references example grids inside its
    # reasoning; parsing the entire text would risk extracting one of those
    # instead of the actual predicted answer.
    answer_text = full_text
    if '</think>' in answer_text:
        answer_text = answer_text.split('</think>', 1)[1]
    predicted_grid = parse_grid(answer_text)

    return {
        'raw_output': full_text,
        'thinking': thinking,
        'predicted': predicted_grid,
        'thinking_length': len(thinking),
    }


# Quick test on first puzzle
print("Testing inference on first puzzle...")
test_result = run_inference(dataset[0], max_new_tokens=4096)
print(f"Thinking length: {test_result['thinking_length']} chars")
print(f"Predicted grid: {test_result['predicted']}")
print(f"Expected grid dims: {grid_dims(dataset[0]['test_output'])}")
if test_result['predicted']:
    print(f"Predicted grid dims: {grid_dims(test_result['predicted'])}")
print(f"\n--- Thinking excerpt (first 800 chars) ---")
print(test_result['thinking'][:800])
print("...")
print(f"\n--- Raw output excerpt (first 400 chars) ---")
print(test_result['raw_output'][:400])

## 7. Evaluation Framework

### Scoring
- **Exact match**: Predicted grid is identical to expected grid (all values and dimensions).
- **Shape match**: Dimensions are correct but values differ.
- **Cell accuracy**: Fraction of cells that match (when dimensions align).

### Metrics
- Per-puzzle-type accuracy
- Overall accuracy
- Average thinking length (reasoning depth)
- Correlation between thinking length and correctness

In [ ]:
def grids_equal(g1, g2):
    """Check if two grids are exactly equal."""
    if g1 is None or g2 is None:
        return False
    if len(g1) != len(g2):
        return False
    for r1, r2 in zip(g1, g2):
        if len(r1) != len(r2):
            return False
        if r1 != r2:
            return False
    return True

def shape_match(g1, g2):
    """Check if grids have same dimensions."""
    if g1 is None or g2 is None:
        return False
    return grid_dims(g1) == grid_dims(g2)

def cell_accuracy(g1, g2):
    """Fraction of matching cells (requires same dims)."""
    if g1 is None or g2 is None:
        return 0.0
    if not shape_match(g1, g2):
        return 0.0
    h, w = grid_dims(g1)
    if h == 0 or w == 0:
        return 0.0
    correct = sum(1 for r in range(h) for c in range(w) if g1[r][c] == g2[r][c])
    return correct / (h * w)

def evaluate_result(result, puzzle):
    """Evaluate a single inference result."""
    expected = puzzle['test_output']
    predicted = result['predicted']
    return {
        'exact_match': grids_equal(predicted, expected),
        'shape_match': shape_match(predicted, expected),
        'cell_accuracy': cell_accuracy(predicted, expected),
        'has_prediction': predicted is not None,
        'thinking_length': result['thinking_length'],
        'predicted_dims': grid_dims(predicted) if predicted else None,
        'expected_dims': grid_dims(expected),
    }

print("Evaluation functions loaded ✓")

## 8. Run Full Benchmark

Run all 36 puzzles. This will take a while due to the reasoning budget (up to 8192 tokens per puzzle).

Progress is printed per puzzle.

In [ ]:
results = []
print(f"Running benchmark: {len(dataset)} puzzles")
print(f"Max tokens per puzzle: {MAX_NEW_TOKENS}")
print("=" * 70)

for i, puzzle in enumerate(dataset):
    t0 = time.time()
    print(f"\n[{i+1}/{len(dataset)}] {puzzle['id']}: {puzzle['description']}")
    
    try:
        result = run_inference(puzzle)
        eval_metrics = evaluate_result(result, puzzle)
        elapsed = time.time() - t0
        
        status = "✓ EXACT" if eval_metrics['exact_match'] else (
            "~ SHAPE" if eval_metrics['shape_match'] else "✗ MISS")
        print(f"  Result: {status} | Cell acc: {eval_metrics['cell_accuracy']:.2%} | "
              f"Think: {eval_metrics['thinking_length']} chars | Time: {elapsed:.1f}s")
        
        results.append({
            'puzzle_id': puzzle['id'],
            'puzzle_type': puzzle['type'],
            'description': puzzle['description'],
            'exact_match': eval_metrics['exact_match'],
            'shape_match': eval_metrics['shape_match'],
            'cell_accuracy': eval_metrics['cell_accuracy'],
            'has_prediction': eval_metrics['has_prediction'],
            'thinking_length': eval_metrics['thinking_length'],
            'predicted_dims': str(eval_metrics['predicted_dims']),
            'expected_dims': str(eval_metrics['expected_dims']),
            'elapsed': elapsed,
            'thinking_excerpt': result['thinking'][:200],
        })
    except Exception as e:
        elapsed = time.time() - t0
        print(f"  ERROR: {e} | Time: {elapsed:.1f}s")
        results.append({
            'puzzle_id': puzzle['id'],
            'puzzle_type': puzzle['type'],
            'description': puzzle['description'],
            'exact_match': False,
            'shape_match': False,
            'cell_accuracy': 0.0,
            'has_prediction': False,
            'thinking_length': 0,
            'predicted_dims': 'None',
            'expected_dims': str(grid_dims(puzzle['test_output'])),
            'elapsed': elapsed,
            'thinking_excerpt': f'ERROR: {str(e)[:200]}',
        })

print("\n" + "=" * 70)
print("Benchmark complete!")

## 9. Results Analysis

Aggregate scores by puzzle type and overall. Visualize performance.

In [ ]:
# Aggregate results
total = len(results)
exact_matches = sum(1 for r in results if r['exact_match'])
shape_matches = sum(1 for r in results if r['shape_match'])
avg_cell_acc = np.mean([r['cell_accuracy'] for r in results])
avg_think = np.mean([r['thinking_length'] for r in results])
total_time = sum(r['elapsed'] for r in results)

print("=" * 70)
print("OVERALL RESULTS")
print("=" * 70)
print(f"Total puzzles:      {total}")
print(f"Exact matches:      {exact_matches}/{total} = {exact_matches/total:.1%}")
print(f"Shape matches:      {shape_matches}/{total} = {shape_matches/total:.1%}")
print(f"Avg cell accuracy:  {avg_cell_acc:.1%}")
print(f"Avg thinking len:   {avg_think:.0f} chars")
print(f"Total time:         {total_time:.0f}s ({total_time/60:.1f} min)")
print()

# Per-type breakdown
print("-" * 70)
print(f"{'Puzzle Type':<30} {'Exact':>6} {'Shape':>6} {'CellAcc':>8} {'AvgThink':>8}")
print("-" * 70)

type_results = {}
for r in results:
    t = r['puzzle_type']
    if t not in type_results:
        type_results[t] = []
    type_results[t].append(r)

for ptype in type_results:
    rs = type_results[ptype]
    em = sum(1 for r in rs if r['exact_match'])
    sm = sum(1 for r in rs if r['shape_match'])
    ca = np.mean([r['cell_accuracy'] for r in rs])
    tl = np.mean([r['thinking_length'] for r in rs])
    desc = rs[0]['description'][:28]
    print(f"{desc:<30} {em:>3}/{len(rs):<3} {sm:>3}/{len(rs):<3} {ca:>7.1%} {tl:>7.0f}")

print("-" * 70)

In [ ]:
# ── Bar chart: Exact match per type ──
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Exact match per type
types_sorted = sorted(type_results.keys(), key=lambda t: sum(1 for r in type_results[t] if r['exact_match']), reverse=True)
em_rates = [sum(1 for r in type_results[t] if r['exact_match']) / len(type_results[t]) for t in types_sorted]
labels = [type_results[t][0]['description'][:20] for t in types_sorted]
axes[0,0].barh(range(len(types_sorted)), em_rates, color='#06d6a0')
axes[0,0].set_yticks(range(len(types_sorted)))
axes[0,0].set_yticklabels(labels, fontsize=8)
axes[0,0].set_xlabel('Exact Match Rate')
axes[0,0].set_title('Exact Match by Puzzle Type')
axes[0,0].set_xlim(0, 1)

# 2. Cell accuracy per type
ca_rates = [np.mean([r['cell_accuracy'] for r in type_results[t]]) for t in types_sorted]
axes[0,1].barh(range(len(types_sorted)), ca_rates, color='#118ab2')
axes[0,1].set_yticks(range(len(types_sorted)))
axes[0,1].set_yticklabels(labels, fontsize=8)
axes[0,1].set_xlabel('Avg Cell Accuracy')
axes[0,1].set_title('Cell Accuracy by Puzzle Type')
axes[0,1].set_xlim(0, 1)

# 3. Thinking length distribution
think_lens = [r['thinking_length'] for r in results]
colors_think = ['#06d6a0' if r['exact_match'] else '#e63946' for r in results]
axes[1,0].bar(range(len(think_lens)), think_lens, color=colors_think)
axes[1,0].set_xlabel('Puzzle Index')
axes[1,0].set_ylabel('Thinking Length (chars)')
axes[1,0].set_title('Reasoning Length per Puzzle (green=correct, red=wrong)')

# 4. Cell accuracy vs thinking length
for r in results:
    color = '#06d6a0' if r['exact_match'] else ('#ffd166' if r['shape_match'] else '#e63946')
    axes[1,1].scatter(r['thinking_length'], r['cell_accuracy'], c=color, s=50, alpha=0.7)
axes[1,1].set_xlabel('Thinking Length (chars)')
axes[1,1].set_ylabel('Cell Accuracy')
axes[1,1].set_title('Reasoning Length vs Accuracy')
axes[1,1].set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.savefig('benchmark_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Results plot saved to benchmark_results.png")

In [ ]:
# ── Detailed results table ──
print(f"{'#':<3} {'Puzzle ID':<25} {'Exact':>6} {'Shape':>6} {'CellAcc':>8} {'Think':>7} {'Time':>6}")
print("=" * 70)
for i, r in enumerate(results):
    em = '✓' if r['exact_match'] else '✗'
    sm = '✓' if r['shape_match'] else '✗'
    print(f"{i+1:<3} {r['puzzle_id']:<25} {em:>6} {sm:>6} {r['cell_accuracy']:>7.1%} {r['thinking_length']:>6.0f}c {r['elapsed']:>5.1f}s")
print("=" * 70)

# ── Show thinking excerpts for first few puzzles ──
print("\n\n=== THINKING EXCERPTS ===")
for i in range(min(5, len(results))):
    r = results[i]
    print(f"\n--- {r['puzzle_id']} ({'CORRECT' if r['exact_match'] else 'WRONG'}) ---")
    print(r['thinking_excerpt'][:300])
    print("...")

In [ ]:
# Save results to JSON
with open('benchmark_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("Results saved to benchmark_results.json")

# Save dataset for reproducibility
dataset_serializable = []
for p in dataset:
    dataset_serializable.append({
        'id': p['id'],
        'type': p['type'],
        'description': p['description'],
        'examples': p['examples'],
        'test_input': p['test_input'],
        'test_output': p['test_output'],
    })
with open('benchmark_dataset.json', 'w') as f:
    json.dump(dataset_serializable, f, indent=2)
print("Dataset saved to benchmark_dataset.json")

# Summary
print("\n" + "=" * 70)
print("BENCHMARK SUMMARY")
print("=" * 70)
print(f"Model: {MODEL_NAME}")
print(f"Puzzles: {len(results)}")
print(f"Types: {len(PUZZLE_TYPES)}")
print(f"Exact Match: {exact_matches}/{total} ({exact_matches/total:.1%})")
print(f"Avg Cell Accuracy: {avg_cell_acc:.1%}")
print(f"Total Time: {total_time/60:.1f} min")
print("=" * 70)

## 10. Conclusion & Notes

### What This Benchmark Tests

This benchmark mimics ARC-AGI 3's core challenge: **inferring abstract transformation rules from few examples and applying them to novel inputs**. The key properties:

1. **Zero instructions** — The model gets no description of the rule, only examples.
2. **Multi-step reasoning** — Each puzzle requires chaining multiple cognitive operations (e.g., identify objects → compute areas → rank → color).
3. **Spatial reasoning** — Grid-based puzzles demand understanding of 2D space, symmetry, connectivity, and paths.
4. **Compositional rules** — Transformations combine simpler operations (e.g., gravity + sorting, counting + conditional reflection).
5. **Rule inference** — The same rule applies across all examples, but the model must figure out what it is.

### Expected Performance

DeepSeek-R1-Distill-Qwen-1.5B is a small reasoning model. Given that frontier models score under 2% on ARC-AGI 3, we expect:
- Very low exact-match rates (likely 0-10%)
- Some shape matches (the model may get dimensions right but values wrong)
- Longer reasoning chains on puzzles it attempts seriously
- Significant variation across puzzle types (simpler rules like color chain may score higher)

### Extending the Benchmark

- Increase `NUM_PUZZLES_PER_TYPE` for more statistical power
- Add more puzzle types (e.g., constraint satisfaction, game-state evaluation)
- Test with larger models (DeepSeek-R1-Distill-Qwen-7B, 32B) for scaling analysis
- Add partial credit scoring (e.g., IoU for object detection puzzles)
- Experiment with different prompt formats (JSON, visual rendering, etc.)